In [7]:
import numpy as np
from scipy.io import wavfile
import IPython.display as ipd
from scipy import signal as sig
from scipy.linalg import solve_toeplitz
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
%config InlineBackend.figure_format = 'svg'

import Funciones as Funciones
from NNMF_GIF import NNMF_GIF
from MetricasRendimiento import MetricasRendimiento
import os
import matplotlib 

# Carga de la señal

In [8]:
with open("Señales Repositorio II Resampleadas/Listas señales/Completa todas.txt", "r") as f:
    archivos = [line.strip() for line in f]
#archivos = archivos[0]

fs_r = 8000
cmap = Funciones.parula_map
color_flujo = 'blue'
color_dflujo = 'red'
color_area = 'orange'
color_darea = 'g'
color_energia = 'magenta'
color_speech = 'black'
legend_loc = 'upper right'



In [9]:
error_1 = []
error_2 = []
etiqueta = []

# Definición de parámetros

In [10]:
norma ='itakura-saito'
pre_iteraciones = 10
iteraciones     = 250
p_filtro        = 12
p = 0.55
pre_enfasis     = True
duracion        = 5.5
guardar_proceso = True

fontsize = 8

In [11]:
archivo = archivos[4]
señales       = np.loadtxt("Señales Repositorio II Resampleadas/"+archivo+".txt", delimiter="\t")
tiempos_      = señales[:, 0]
señal_speech_ = señales[:, 1]
señal_flujo_  = señales[:, 2]
señal_dflujo_ = señales[:, 3]
señal_area_   = señales[:, 4]
señal_darea_  = señales[:, 5]
to = 750
tf = 800
no = int(np.round(to *fs_r/1000)) 
nf = int(np.round(tf * fs_r/1000))
tiempos       = tiempos_[no:nf]

señal_speech  = señal_speech_[no:nf]
señal_flujo   = señal_flujo_[no:nf]
señal_dflujo   = señal_dflujo_[no:nf]
señal_area    = señal_area_[no:nf]
señal_darea   = señal_darea_[no:nf]

SP1 = NNMF_GIF(señal_speech,tiempos,fs_r,
        norma=norma,
        pre_iteraciones=pre_iteraciones,
        iteraciones=iteraciones,
        p_filtro=p_filtro,
        p=p,
        pre_enfasis=pre_enfasis,
        duracion=duracion,
        guardar_proceso=guardar_proceso)

señal_flujo  = señal_flujo[SP1.orden_filtro_tracto-1:len(señal_flujo)-SP1.orden_filtro_tracto-1]
señal_dflujo = señal_dflujo[SP1.orden_filtro_tracto-1:len(señal_dflujo)-SP1.orden_filtro_tracto-1]


# Gráficas

In [12]:
#os.makedirs("Prueba Repositorio II/Animacion/Itakura/"+archivo)
for i in range(0,iteraciones+1):
      #i=1
      fw1,hw1 = sig.freqz([1],SP1.proceso_a_W1[i],fs=fs_r)
      fw2,hw2 = sig.freqz([1],SP1.proceso_a_W2[i],fs=fs_r)


      tiempo_flujo1,_,flujo1 = Funciones.Sincronizar2(SP1.tiempos_gve,señal_flujo,SP1.proceso_flujo1[i])
      tiempo_flujo2,_,flujo2 = Funciones.Sincronizar2(SP1.tiempos_gve,señal_flujo,SP1.proceso_flujo2[i])

      tiempo_dflujo1,señal_dflujo_1,dflujo1 = Funciones.Sincronizar2(SP1.tiempos_gve,señal_dflujo,SP1.proceso_dflujo1[i])
      tiempo_dflujo2,señal_dflujo_2,dflujo2 = Funciones.Sincronizar2(SP1.tiempos_gve,señal_dflujo,SP1.proceso_dflujo2[i])

      MR1 = MetricasRendimiento(señal_dflujo_1,Funciones.Escalar(dflujo1,señal_dflujo_1),fs_r,tiempo_dflujo1)
      MR2 = MetricasRendimiento(señal_dflujo_2,Funciones.Escalar(dflujo2,señal_dflujo_2),fs_r,tiempo_dflujo2)

      color2 = 'blue'
      color1 = 'red'
      color_error = 'green'
      loc = 'lower right'
      fftseñal,fseñal = Funciones.FFT(señal_speech,fs_r)

      fig = plt.figure(figsize=(13,9))
      ''' 
      widths  = [0.6,0.6,1,0.5]
      heights = [1,# Espectrograma original 
            1,# Espectrograma aproximado
            1,# Plot  de H
            ]
      gs  = fig.add_gridspec(3, 4,width_ratios=widths,height_ratios=heights,left=0,right=1,wspace=0.15,hspace=0.35)
      # Espectrograma original
      ax0 = fig.add_subplot(gs[0,0:2])
      # Espectrograma aproximado
      ax1 = fig.add_subplot(gs[1,0:2])
      # Plot  de H
      ax2 = fig.add_subplot(gs[2, 0:2])
      # Plot de W
      ax3 = fig.add_subplot(gs[0,2])
      # Polos y ceros
      ax4 = fig.add_subplot(gs[0,3])
      # Flujos
      ax5 = fig.add_subplot(gs[1,2:4])
      # Derivadas
      ax6 = fig.add_subplot(gs[2,2:4])
      '''
      widths  = [0.6,0.6,0.6,0.6,0.6,0.6]
      heights = [1,# Espectrograma original 
            1,# Espectrograma aproximado
            1,# Plot  de H
            0.55
            ]
      gs  = fig.add_gridspec(4, 6,width_ratios=widths,height_ratios=heights,wspace=0.25,hspace=0.45)
      # Espectrograma original
      ax0 = fig.add_subplot(gs[0,0:3])
      # Espectrograma aproximado
      ax1 = fig.add_subplot(gs[1,0:3])
      # Plot  de H
      ax2 = fig.add_subplot(gs[2, 0:3])
      # Plot de W
      ax3 = fig.add_subplot(gs[0,3:5])
      # Polos y ceros
      ax4 = fig.add_subplot(gs[0,5:6])
      #ax4 = fig.add_subplot(gs[0,5:6])
      # Flujos
      ax5 = fig.add_subplot(gs[1,3:6])
      # Derivadas
      ax6 = fig.add_subplot(gs[2,3:6])
      # Error X
      ax7 = fig.add_subplot(gs[3,0:2])
      # Error W
      ax8 = fig.add_subplot(gs[3,2:4])
      # Error H
      ax9 = fig.add_subplot(gs[3,4:6])
      # ===============================  Espectrograma original ==============================================================================================

      espect = ax0.imshow(SP1.espectrograma,cmap = cmap,origin='lower',aspect='auto',extent=[SP1.to_espec,SP1.tf_espec,SP1.fo_espec,SP1.ff_espec]);
      ax0.set_ylabel(r"$f$ [Hz]",fontsize=fontsize);
      ax0_ = ax0.twinx()
      ax0_.plot(SP1.tiempos_espec,señal_darea[SP1.long_ventana_espect-1:],color='white',label=r"$\dfrac{dA(t)}{dt}$",linewidth=2);
      ax0_.legend(loc='upper right',fontsize=fontsize);
      ax0_.set_xlim(SP1.to_espec,SP1.tf_espec)
      ax0_.set_xticks(np.arange(SP1.to_espec,SP1.tf_espec,SP1.duracion_ventana))
      ax0_.set_yticks([])
      ax0.set_xlabel('Tiempo [ms]',fontsize=fontsize);
      ax0.tick_params(axis='x', labelsize=fontsize)
      ax0.tick_params(axis='y',labelsize=fontsize)

      # ===============================  Espectrograma aproximado ==============================================================================================


      espect = ax1.imshow(SP1.proceso_espectrograma_aprox[i],cmap = cmap,origin='lower',aspect='auto',extent=[SP1.to_espec,SP1.tf_espec,SP1.fo_espec,SP1.ff_espec]);
      ax1.set_ylabel(r"$f$ [Hz]",fontsize=fontsize);
      ax1_ = ax1.twinx()
      ax1_.plot(tiempos[SP1.long_ventana_espect:],señal_darea[SP1.long_ventana_espect:],color='white',label=r"$\dfrac{dA(t)}{dt}$",linewidth=2);
      ax1_.legend(loc='upper right',fontsize=fontsize);
      ax1_.set_xlim(SP1.to_espec,SP1.tf_espec)
      ax1_.set_xticks(np.arange(SP1.to_espec,SP1.tf_espec,SP1.duracion_ventana))
      ax1_.set_yticks([])
      ax1.set_xlabel('Tiempo [ms]',fontsize=fontsize);
      ax1.tick_params(axis='y',labelsize=fontsize)
      ax1.tick_params(axis='x',labelsize=fontsize)

      # ===============================  Funciones de activación  ==============================================================================================

      ax2.plot(SP1.tiempos_espec,SP1.proceso_H1[i],color=color1,label=r'$H_1$')
      ax2.plot(SP1.tiempos_espec,SP1.proceso_H2[i],color=color2,label=r'$H_2$')
      ax2.set_xlim(SP1.to_espec,SP1.tf_espec)
      ax2.set_xticks(np.arange(SP1.to_espec,SP1.tf_espec,SP1.duracion_ventana))
      ax2.set_xticks(np.arange(SP1.to_espec,SP1.tf_espec,SP1.duracion_ventana/2),minor=True)
      ax2.grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
      ax2.legend(loc=loc,fontsize=fontsize);
      ax2.set_xlabel('Tiempo [ms]',fontsize=fontsize);
      ax2.tick_params(axis='x',labelsize=fontsize)
      ax2.tick_params(axis='y',labelsize=fontsize)

      # ===============================  Respuesta en frecuencia de W1 y W2 ==============================================================================================

      ax3.plot(fw1,10*np.log10(np.abs(hw1)),label=r'$W_1$',color=color1)
      ax3.plot(fw2,10*np.log10(np.abs(hw2)),label=r'$W_2$',color=color2)
      ax3.plot(fseñal,10*np.log10(np.abs(fftseñal)),label='speech',color=color_speech,zorder=-1)
      ax3.set_xlim(0,fs_r/2);
      #ax3.set_xticks(np.arange(0,fs_r/2,100),minor=True)
      ax3.grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
      ax3.set_xlabel('Frecuencia [Hz]',fontsize=fontsize);
      ax3.legend(loc='upper right',fontsize=fontsize);
      ax3.tick_params(axis='x',labelsize=fontsize)
      ax3.tick_params(axis='y',labelsize=fontsize)
      # ===============================  Polos y ceros ==============================================================================================

      theta = np.linspace(0, 2*np.pi, 200)

      ax4.plot(np.cos(theta), np.sin(theta), 'k--', alpha=1)
      ax4.yaxis.tick_right()
      ax4.yaxis.set_label_position("right")
      # Graficar ceros (O) y polos (X)
      ax4.scatter(np.real(SP1.proceso_ceros_W1[i]), np.imag(SP1.proceso_ceros_W1[i]), s=80, facecolors='none', edgecolors=color1, label='Ceros')
      ax4.scatter(np.real(SP1.proceso_polos_W1[i]), np.imag(SP1.proceso_polos_W1[i]), s=80, marker='x', color=color1, label='Polos')

      # Graficar ceros (O) y polos (X)
      ax4.scatter(np.real(SP1.proceso_ceros_W2[i]), np.imag(SP1.proceso_ceros_W2[i]), s=80,linestyle='dashed', facecolors='none', edgecolors=color2, label='Ceros')
      ax4.scatter(np.real(SP1.proceso_polos_W2[i]), np.imag(SP1.proceso_polos_W2[i]), s=80, marker='x', color=color2, label='Polos')
      ax4.set_xticks(np.arange(-1.25,1.25,0.25),minor=True)
      ax4.set_yticks(np.arange(-1.25,1.25,0.25),minor=True)

      ax4.set_ylabel(r"$Im\{z\}$",fontsize=fontsize)
      #ax5_.set_ylim([-1.75,1.75])
      ax4.set_xlabel(r"$Re\{z\}$",fontsize=fontsize)
      ax4.grid(True,linestyle='dashed',color='gray',alpha=0.3)
      ax4.tick_params(axis='x',labelsize=fontsize)
      ax4.tick_params(axis='y',labelsize=fontsize)

      # ===============================  Flujos glóticos ==============================================================================================
      ax5.plot(SP1.tiempos_gve,señal_flujo,color=color_speech,label=r'$u_g(t)$');
      ax5.plot(tiempo_flujo1,flujo1,color=color1,label=r'$\hat{u}_{g1}(t)$');
      ax5.plot(tiempo_flujo2,flujo2,color=color2,label=r'$\hat{u}_{g2}(t)$');
      ax5.set_xlim([to,tf]);
      #ax5.set_ylim([-1.15,1.15]);
      ax5.set_xticks(np.arange(to,tf,2.5),minor=True)
      ax5.set_xlabel("Tiempo [ms]",fontsize=fontsize)
      ax5.grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
      ax5.legend(loc=loc,fontsize=fontsize);
      ax5.yaxis.tick_right()
      ax5.yaxis.set_label_position("right")
      ax5.tick_params(axis='x',labelsize=fontsize)
      ax5.tick_params(axis='y',labelsize=fontsize)

      # ===============================  Funciones glóticas  ==============================================================================================

      ax6.plot(tiempo_dflujo1,dflujo1,color=color1,label=r'$\hat{v}_{g1}(t)$'+str(' ~ ')+r'$E_{vg}$'+f'={MR1.Error:.7f}');
      ax6.plot(tiempo_dflujo2,dflujo2,color=color2,label=r'$\hat{v}_{g2}(t)$'+str(' ~ ')+r'$E_{vg}$'+f'={MR2.Error:.7f}');
      ax6.plot(SP1.tiempos_gve,señal_dflujo,color=color_speech,label=r'$v_g(t)$');
      ax6.set_xlim([to,tf]);
      ax6.set_xticks(np.arange(to,tf,2.5),minor=True)
      ax6.grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
      ax6.legend(loc=loc,fontsize=fontsize);
      ax6.set_xlim([to,tf])
      #ax6.set_ylim([-1.15,1.15]);
      ax6.tick_params(axis='x', which='both')
      ax6.set_xlabel("Tiempo [ms]",fontsize=fontsize)
      ax6.yaxis.tick_right()
      ax6.yaxis.set_label_position("right")
      ax6.tick_params(axis='x',labelsize=fontsize)
      ax6.tick_params(axis='y',labelsize=fontsize)
      # ===============================  Error X  ==============================================================================================
      iteraciones_X = np.arange(0,iteraciones+1)
      lw=1
      if i == 0:
            ax7.scatter(iteraciones_X[i],SP1.error_NNMF[i], s=lw,color='orange',label=f"$ΔX({i})$={SP1.error_NNMF[i]:.7f}")
      else:
            ax7.plot(iteraciones_X[0:i+1],SP1.error_NNMF[0:i+1],color='orange',label=f"$ΔX({i})$={SP1.error_NNMF[i]:.7f}")
      #ax7.plot(SP1.iteraciones,SP1.error_NNMF[SP1.iteraciones-1],color='orange')
      ax7.set_xlabel("Iteración n°",fontsize=fontsize)
      ax7.set_yticks([])
      ax7.set_xlim(-1,i+1)
      #ax7.set_ylabel("ΔX")
      #ax7.grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
      ax7.legend(loc='upper right',fontsize=fontsize);
      ax7.tick_params(axis='both',labelsize=fontsize)
      # ===============================  Error W  ==============================================================================================
      lw=1
      if i == 1:
            ax8.scatter(iteraciones_X[1],SP1.dif_error_W1[0], s=lw,color=color1,label=f"$ΔW_1({i})$={SP1.dif_error_W1[i]:.7f}")
            ax8.scatter(iteraciones_X[1],SP1.dif_error_W2[0], s=lw,color=color2,label=f"$ΔW_2({i})$={SP1.dif_error_W2[i]:.7f}")
      if i >1:
            ax8.plot(iteraciones_X[1:i+1],SP1.dif_error_W1[0:i] ,color=color1,label=f"$ΔW_1({i})$={SP1.dif_error_W1[i]:.7f}")
            ax8.plot(iteraciones_X[1:i+1],SP1.dif_error_W2[0:i],color=color2,label=f"$ΔW_2({i})$={SP1.dif_error_W2[i]:.7f}")
      #ax7.plot(SP1.iteraciones,SP1.error_NNMF[SP1.iteraciones-1],color='orange')
      ax8.set_xlabel("Iteración n°",fontsize=fontsize)
      ax8.set_yticks([])
      #ax7.set_ylabel("ΔX")
      #ax7.grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
      ax8.legend(loc='upper right',fontsize=fontsize);
      ax8.tick_params(axis='both',labelsize=fontsize)
      # ===============================  Error H  ==============================================================================================
      lw=1
      if i == 1:
            ax9.scatter(iteraciones_X[1],SP1.dif_error_H1[0], s=lw,color=color1,label=f"$ΔH_1({i})$={SP1.dif_error_H1[i]:.7f}")
            ax9.scatter(iteraciones_X[1],SP1.dif_error_H2[0], s=lw,color=color2,label=f"$ΔH_2({i})$={SP1.dif_error_H2[i]:.7f}")
      if i >1:
            ax9.plot(iteraciones_X[1:i+1],SP1.dif_error_H1[0:i] ,color=color1,label=f"$ΔH_1({i})$={SP1.dif_error_H1[i]:.7f}")
            ax9.plot(iteraciones_X[1:i+1],SP1.dif_error_H2[0:i],color=color2,label=f"$ΔH_2({i})$={SP1.dif_error_H2[i]:.7f}")
      #ax7.plot(SP1.iteraciones,SP1.error_NNMF[SP1.iteraciones-1],color='orange')
      ax9.set_xlabel("Iteración n°",fontsize=fontsize)
      ax9.set_yticks([])
      #ax7.set_ylabel("ΔX")
      #ax7.grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
      ax9.legend(loc='upper right',fontsize=fontsize);
      ax9.tick_params(axis='both',labelsize=fontsize)


      fig.suptitle(
      f"{archivo} | norma: {norma} | pre-iteraciones: {pre_iteraciones} | iteración n°: {i}",
      fontsize=8,
      y=0.91
      );
      fig.savefig("Prueba Repositorio II/Animacion/Itakura/"+archivo+"/"+str(i)+'.pdf',bbox_inches = 'tight');
      plt.close();


No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
